# NYC TLC parquet → Azure Blob (Google Colab)

Streams NYC TLC trip-record parquet files **straight from the TLC URL into an Azure Blob container** — no files are written to the Colab disk.

**How to use**
1. Run **Cell 1** to install the Azure SDK.
2. Add your connection string as a Colab secret named `AZURE_STORAGE_CONNECTION_STRING` (🔑 icon in the left sidebar, toggle *Notebook access* on). Or skip it and you'll be prompted securely in Cell 3.
3. Set options in **Cell 2** (record type, year range, container, prefix).
4. Run the remaining cells top-to-bottom.

In [1]:
# Cell 1 — install dependencies (requests is already in Colab)
!pip install -q azure-storage-blob

In [7]:
# Cell 2 — configuration  (edit these values)
RECORD_TYPES = ["green"]   #@param  one or more of: yellow, green, fhv, fhvhv
START_YEAR   = 2022        #@param {type:"integer"}
END_YEAR     = 2026        #@param {type:"integer"}
CONTAINER    = "greentaxi"  #@param {type:"string"}
BLOB_PREFIX  = "raw/"         #@param {type:"string"}
WORKERS      = 6           #@param {type:"integer"}
OVERWRITE    = True       #@param {type:"boolean"}

In [12]:
# Cell 3 — get the connection string (Colab secret first, else secure prompt)
import os
import json

#CONN = None
#try:
#    from google.colab import userdata
#    CONN = userdata.get('AZURE_STORAGE_CONNECTION_STRING')
#except Exception:
#    pass
#
#if not CONN:
#    CONN = os.environ.get('AZURE_STORAGE_CONNECTION_STRING')
#
#if not CONN:
#    import getpass
#    CONN = getpass.getpass('Paste AZURE_STORAGE_CONNECTION_STRING: ')

#assert CONN, 'No connection string provided.'
#print('Connection string loaded (%d chars).' % len(CONN))


def load_config_azure(config_path="config.json"):
    """Load the Azure configuration parameters from the config.json file."""
    with open(config_path, "r", encoding="utf-8") as config_file:
        config = json.load(config_file)
    return config["AZURE_CONNECTION_STRING"], config["CONTAINER_NAME"]

AZURE_CONNECTION_STRING, CONTAINER_NAME = load_config_azure()
CONN =  AZURE_CONNECTION_STRING





In [13]:
# Cell 4 — the scraper (streams URL -> blob, nothing touches local disk)
import requests
from concurrent.futures import ThreadPoolExecutor, as_completed
from azure.storage.blob import BlobServiceClient

BASE_URL = "https://d37ci6vzurychx.cloudfront.net/trip-data"

def iter_months(start_year, end_year):
    for year in range(start_year, end_year + 1):
        for month in range(1, 13):
            yield year, month

def stream_to_azure(record_type, year, month, container_client, prefix, session, overwrite):
    fname = f"{record_type}_tripdata_{year:04d}-{month:02d}.parquet"
    url = f"{BASE_URL}/{fname}"
    blob_name = f"{prefix}{fname}"
    blob_client = container_client.get_blob_client(blob_name)
    try:
        if not overwrite and blob_client.exists():
            return "skip-exists", blob_name
        with session.get(url, stream=True, timeout=300) as resp:
            # CloudFront/S3 returns 403 (not 404) for files TLC hasn't published yet.
            if resp.status_code in (403, 404):
                return "missing", blob_name
            resp.raise_for_status()
            resp.raw.decode_content = True
            length = resp.headers.get("Content-Length")
            blob_client.upload_blob(
                resp.raw,
                overwrite=True,
                length=int(length) if length else None,
                max_concurrency=1,
                content_type="application/vnd.apache.parquet",
            )
        mb = int(length) / (1 << 20) if length else 0
        return f"ok ({mb:.1f} MB)", blob_name
    except Exception as exc:
        return f"error: {exc}", blob_name

def run(types, start_year, end_year, container, prefix, workers, overwrite, conn):
    service = BlobServiceClient.from_connection_string(conn)
    container_client = service.get_container_client(container)
    try:
        container_client.create_container()
        print(f"Created container '{container}'.")
    except Exception:
        pass  # already exists

    jobs = [(t, y, m) for t in types for (y, m) in iter_months(start_year, end_year)]
    print(f"Planning {len(jobs)} files: types={types}, {start_year}-{end_year} "
          f"-> azure://{container}/{prefix}\n")

    session = requests.Session()
    session.headers.update({"User-Agent": "tlc-parquet-scraper/colab"})
    counts = {"ok": 0, "skip-exists": 0, "missing": 0, "error": 0}

    with ThreadPoolExecutor(max_workers=workers) as pool:
        futures = {pool.submit(stream_to_azure, t, y, m, container_client,
                               prefix, session, overwrite): (t, y, m)
                   for (t, y, m) in jobs}
        for fut in as_completed(futures):
            status, name = fut.result()
            key = ("ok" if status.startswith("ok")
                   else "error" if status.startswith("error") else status)
            counts[key] = counts.get(key, 0) + 1
            print(f"  [{status:>16}] {name}")

    print("\nDone.")
    print(f"  uploaded        : {counts['ok']}")
    print(f"  already present : {counts['skip-exists']}")
    print(f"  not yet published: {counts['missing']}")
    print(f"  errors          : {counts['error']}")
    return counts

In [14]:
# Cell 5 — run it
counts = run(RECORD_TYPES, START_YEAR, END_YEAR, CONTAINER,
             BLOB_PREFIX, WORKERS, OVERWRITE, CONN)

Planning 60 files: types=['green'], 2022-2026 -> azure://greentaxi/raw/

  [     ok (1.4 MB)] raw/green_tripdata_2022-02.parquet
  [     ok (1.5 MB)] raw/green_tripdata_2022-03.parquet
  [     ok (1.3 MB)] raw/green_tripdata_2022-07.parquet
  [     ok (1.3 MB)] raw/green_tripdata_2022-08.parquet
  [     ok (1.5 MB)] raw/green_tripdata_2022-05.parquet
  [     ok (1.4 MB)] raw/green_tripdata_2022-10.parquet
  [     ok (1.5 MB)] raw/green_tripdata_2022-06.parquet
  [     ok (1.4 MB)] raw/green_tripdata_2022-09.parquet
  [     ok (1.2 MB)] raw/green_tripdata_2022-01.parquet
  [     ok (1.5 MB)] raw/green_tripdata_2022-04.parquet
  [     ok (1.2 MB)] raw/green_tripdata_2022-11.parquet
  [     ok (1.4 MB)] raw/green_tripdata_2022-12.parquet
  [     ok (1.7 MB)] raw/green_tripdata_2023-03.parquet
  [     ok (1.4 MB)] raw/green_tripdata_2023-01.parquet
  [     ok (1.5 MB)] raw/green_tripdata_2023-04.parquet
  [     ok (1.6 MB)] raw/green_tripdata_2023-05.parquet
  [     ok (1.5 MB)] raw/green_

In [6]:
# Cell 6 (optional) — verify what landed in the container
from azure.storage.blob import BlobServiceClient
svc = BlobServiceClient.from_connection_string(CONN)
cc = svc.get_container_client(CONTAINER)
blobs = list(cc.list_blobs(name_starts_with=BLOB_PREFIX))
total_mb = sum(b.size for b in blobs) / (1 << 20)
print(f"{len(blobs)} blobs under '{BLOB_PREFIX}', {total_mb:.1f} MB total\n")
for b in blobs:
    print(f"  {b.name}  ({b.size/(1<<20):.1f} MB)")

53 blobs under 'raw/', 68.2 MB total

  raw/green_tripdata_2022-01.parquet  (1.2 MB)
  raw/green_tripdata_2022-02.parquet  (1.4 MB)
  raw/green_tripdata_2022-03.parquet  (1.5 MB)
  raw/green_tripdata_2022-04.parquet  (1.5 MB)
  raw/green_tripdata_2022-05.parquet  (1.5 MB)
  raw/green_tripdata_2022-06.parquet  (1.5 MB)
  raw/green_tripdata_2022-07.parquet  (1.3 MB)
  raw/green_tripdata_2022-08.parquet  (1.3 MB)
  raw/green_tripdata_2022-09.parquet  (1.4 MB)
  raw/green_tripdata_2022-10.parquet  (1.4 MB)
  raw/green_tripdata_2022-11.parquet  (1.2 MB)
  raw/green_tripdata_2022-12.parquet  (1.4 MB)
  raw/green_tripdata_2023-01.parquet  (1.4 MB)
  raw/green_tripdata_2023-02.parquet  (1.5 MB)
  raw/green_tripdata_2023-03.parquet  (1.7 MB)
  raw/green_tripdata_2023-04.parquet  (1.5 MB)
  raw/green_tripdata_2023-05.parquet  (1.6 MB)
  raw/green_tripdata_2023-06.parquet  (1.5 MB)
  raw/green_tripdata_2023-07.parquet  (1.4 MB)
  raw/green_tripdata_2023-08.parquet  (1.4 MB)
  raw/green_tripdata_2